### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is used useful for the following: 

* Tracking agent behavior with logging, analytics and debugging. 
* Transforming prompts, tool selection, and output formatting. 
* Adding retries, fallbacks, and early termination logic.
* Applying root limits omar omar guardrails and PII Addiction

#### init

In [11]:
import os

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

os.environ["OPENAI_BASE_URL"] = "http://localhost:4000"
os.environ["OPENAI_API_KEY"] = "sk_dummy_key"

#### Summarization Middleware

Auto-summarizes conversation history when approaching rate limits, preserving recent messages while compressing old context

##### Message Count Trigger

In [12]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

# Message-based summarization

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ]
)



In [ ]:
# First we need to create a thread, with its thread id
config={"configurable": {"thread_id":"test-1"}}

# Test Data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?"
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")



##### Token Count Trigger

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

@tool
def search_hotel(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent = create_agent(
    model="gpt-4o",
    tools=[search_hotel],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o",
            trigger=("tokens", 550),
            keep=("tokens", 200)
        )
    ]
)

In [ ]:
config = {"configurable": {"thread_id": "test_1"}}

# Token Counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars = 1 token

In [ ]:
# Run test

cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke({"messages": [HumanMessage(content=f"find hotels in {city}")]}, config=config)

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response["messages"])} messages")
    print(response["messages"])

##### Fraction-based Trigger

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

@tool
def search_hotel(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}: Grand Hotel $350/night, City Inn $180/night, Budget Stay $75/night"""

agent = create_agent(
    model="gpt-4o",
    tools=[search_hotel],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o",
            trigger=("fraction", 0.005), # 0.5% = ~640 tokens
            keep=("fraction", 0.002) # 0.2% = ~256 tokens (I think this is specific to the context limit of the model used.)
        )
    ]
)

In [ ]:
config = {"configurable": {"thread_id": "test_1"}}

# Token Counter (approximate)
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Run test

cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke({"messages": [HumanMessage(content=f"find hotels in {city}")]}, config=config)

    tokens = count_tokens(response["messages"])
    fraction = tokens / 1048576
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response["messages"])} messages")
    print(response["messages"])

### Human-In-The-Loop Middleware

Pause agent execution for human approval, editing or rejection of tool calls before they execute. Useful for:
* High-stakes operations(e.g. database modification)
* Lengthy conversations where human feedback guides the agent
* Workflows where human oversight is necessary

In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

# tools

def read_email(email_id: str) -> str:
    """Mock function to read an email by its ID"""
    return f"Email content for ID: {email_id}"

def send_email(recipient: str, subject: str, body: str):
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject '{subject}'"


In [27]:
agent = create_agent(
    model = "groq:openai/gpt-oss-120b",
    tools = [read_email, send_email],
    checkpointer = InMemorySaver(),
    system_prompt = "Don't markdown or '\\' text formatting(like \\n or \\t) in your responses.",
    middleware= [
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "send_email": {
                    "allowed_decisions": ['approved', 'edit', 'reject']
                },
                "read_email": False
            }
        )
    ]
)

In [28]:
config = {"configurable": {"thread_id": "testid_1"}}

# First Step: Request

result = agent.invoke(
    {"messages": HumanMessage(content="Send an email to john@doe.com talking about french fries")},
    config=config
)



# Second Step: Human Decision

if "__interrupt__" in result:
    print("Paused! Waiting for feedback...")

    print("1. Approve | 2. Reject | 3. Edit ")
    choice = int(input("Choice: "))

    if choice == 1:
        result = agent.invoke(
            Command(
                resume={
                    "decisions": {
                        "type": "approved"
                    }
                }
            )
        )

        print(f"Response: {result["messages"][-1].content}")
    elif choice == 2:
        result = agent.invoke(
            Command(
                resume={
                    "decisions": {
                        "type": "reject"
                    }
                }
            )
        )
    elif choice == 3:
        result = agent.invoke(
            Command(
                resume={
                    "decisions": {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email",
                            "args": {
                                "recipient": "jane@doe.com",
                                "subject": ""
                            }
                        }
                    }
                }
            )
        )

Paused! Waiting for feedback...
1. Approve | 2. Reject | 3. Edit 


ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id